# pyuvis

> This package provides tools to work with Cassini UVIS data.

## How to install

It is recommended to manage your Python environments with `conda` or its new and faster sibling [mamba](https://mamba.readthedocs.io/en/latest/installation.html).

After installation of `mamba`/`conda`, create a new Python environment with the name `py38` (or whatever name you prefer) with:

```
conda create -n py38 python=3.8 -c conda-forge
```

> Note: If you installed mambaforge in the previous step, the `conda-forge` channel is the default source for conda packages and you wouldn't need to add it here.
If you need a refresher or intro into managing your Python environment with conda, have a look at my tutorials [here (long version)](https://michaelaye.github.io/talk/how-to-live-peacefully-with-ana/mini/conda/) and [here (short version)](https://michaelaye.github.io/talk/best-practices-for-conda-based-python-science-analysis-environments/).

Once you have created the new environment, activate it:

```
conda activate py38
```
(or your \<env> name)

and install the latest requirements for the package:

```
mamba install -c conda-forge hvplot xarray pandas
```

followed by a pip install for this package (a `pyuvis` conda package will be created later):

```
pip install pyuvis
```

The package depends on `planetarypy` to organize data discovery, download, and local data management and the pip install should pull in `planetarypy` with the minimum version of 0.15.

## How to use

When launched for the first time, the `planetarypy` package will ask for a general storage path for all data managed by `planetarypy`.
You should give it a folder path where you have sufficient space to store the planetary data you want to work with.

The different available modules were born out of special use cases for analysis of UVIS data, but for the general user the most important classes are `UVPDS` and the `UVISObs`.

### UVPDS

The UVPDS class is designed to deal with exactly one PDS product for either EUV or FUV data:

In [ ]:
from pyuvis import UVPDS

In [ ]:
pid = "FUV2005_172_03_35"
uv = UVPDS(pid)

If the data file does not exist on your local storage yet, `planetarypy` will go ahead and download it for you and provide the local path to UVPDS for reading it.

You don't have to do anything to get the data and you don't have to decide anything about where to store the data. :)
If you are interested in seeing how the `planetarypy.uvis` module achieves this, have a look at https://michaelaye.github.io/nbplanetary/cassini_uvis

While the data qube itself is accessible as a `numpy.array` via the `data` attribute:

In [ ]:
uv.data[100]

... it is recommended to use the `xarray` attribute for a more advanced data container with labeled coordinate axes 

`("spectral", "spatial", "samples")`
:

In [ ]:
uv.xarray

One can see that the spatial range was already reduced to the valid range of 2..61 according to the label data.

> Note that the sample count starts at 1, to be fitting to the UVIS manual.

I usually story the `xarray.DataArray` into a `da` or `arr` variable for further analysis and plotting:

In [ ]:
arr = uv.xarray

A built-in example `plot` method shows you how to use `hvplot` to generate interactive plots:

In [ ]:
uv.plot()

The widget to the right allows you to scroll through the `samples` dimension.

> Hint: You can look at the source code of any method using the double question mark:

In [ ]:
uv.plot??

### Plot style

I chose to use the less precise (due to our non-equidistant wavelengths) raster plot by default, instead of the more precise quadmesh, because the latter creates an annoying aliasing effect,possibly due to gridlines, I'm am investigating.

If you need to be really precise on the wavelength position, use the `precise=True` keyword:

In [ ]:
uv.plot(precise=True)

Control the visual limits using the percentile stretch or the `clim` parameter directly:

In [ ]:
uv.plot(percentiles=(2, 98))

In [ ]:
uv.plot(clim=(None, 10))

Control the colormap using `cmap`:

In [ ]:
uv.plot(precise=False, cmap='magma')

All the options added to the `UVPDS.plot` method can be applied to the xarray directly, if you need the direct control.

For example, you can select a specific sample using the `xarray.sel` method and plot only that:

In [ ]:
arr.sel(samples=100).hvplot(x='spectral', y='spatial', cmap='viridis')

One can also apply a video scrubber to the samples instead:

In [ ]:
np.percentile(arr, (1, 99))  # determine good color stretch for inner 98% of the data

In [ ]:
arr.hvplot(
    x="spectral",
    y="spatial",
    widget_type="scrubber",
    widget_location="bottom",
    clim=(0, 55),
    cmap="viridis",
)

### Calibrated data

Calibration masks are being downloaded automatically by the `planetarypy.uvis` module.

It is applied following the qube description in the calibration data label:

In [ ]:
uv.caliblabel.QUBE.DESCRIPTION

In [ ]:
uv.calibrated

Plot the calibrated data by using the `calibrated=True` option to `plot()`:

In [ ]:
uv.plot(calibrated=True, clim=(0, None))

### Labels
One can get to the labels via the `datalabel` or `caliblabel` attribute:

In [ ]:
uv.caliblabel.SLIT_STATE

In [ ]:
uv.datalabel.SC_PLANET_POSITION_VECTOR

In [ ]:
uv.cal_data.data[uv.cal_data.data == -1] = np.nan

## `UVISObs`

The `UVISObs` class is just a convenience class that provides quick access to both the EUV and FUV version of a UVIS Observation with the same start time.

One can conveniently initiate it with only the `yyyy_jjj_hh_mm` time string or with the full PDS product_id of either a FUV or EUV file:

In [ ]:
from pyuvis import UVISObs

In [ ]:
pid

In [ ]:
obs = UVISObs(pid)

In [ ]:
obs.euv.datalabel.PRODUCT_ID

In [ ]:
obs.fuv.datalabel.PRODUCT_ID

### hvplot / holoviews

`hvplot` is an entry interface to the `holoviews` plotting framework.
One of the design principles of `holoviews` is re-usability, which makes combining existing plots a breeze.

What happens below is that '+'-ing two plots creates a `Layout` object consisting of the two `DynamicMap` objects, `dynamic` because of the scroll widget through the samples.

This layout object can now be rendered with additional options:

In [ ]:
layout = obs.euv.plot() + obs.fuv.plot()
print(layout)

In [ ]:
layout

In [ ]:
layout.cols(1)

## Found issues?

Please report any issues found at the GitHub issue tracker for this project: 

https://github.com/Cassini-UVIS/pyuvis/issues